# Shape Mismatch Leak — yudasong/Reinforcement-Learning-Branch-and-Bound

**Smell (`othello/keras/OthelloNNet.py` line 25):** After 4 convolutional layers, `Flatten()` is applied to `h_conv4` and then immediately passed to `Dense(1024)`. This creates a massive `(board_x-4)*(board_y-4)*num_channels × 1024` weight matrix. For an 8×8 Othello board that's `4*4*256 × 1024 = 4M+ parameters` just in that one layer.

**Fix:** Replace `Flatten() → Dense(1024)` with `GlobalAveragePooling2D() → Dense(256)`. GAP averages over the spatial dimensions, producing a `num_channels`-d vector — far fewer parameters and less memory.

In [ ]:
!pip install -q codecarbon

In [ ]:
import codecarbon, os, json
_dir = os.path.join(os.path.dirname(codecarbon.__file__), 'data', 'private_infra')
os.makedirs(_dir, exist_ok=True)
_p = os.path.join(_dir, 'nordic_emissions.json')
if not os.path.exists(_p):
    with open(_p, 'w') as f: json.dump({}, f)
print('CodeCarbon ready')

In [ ]:
import gc, os, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.layers import (Input, Conv2D, BatchNormalization, Activation,
                                      Flatten, Dense, Dropout, GlobalAveragePooling2D)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from codecarbon import EmissionsTracker

os.makedirs('results', exist_ok=True)
N_RUNS      = 10
EPOCHS      = 5
BATCH       = 64
BOARD_X     = 8
BOARD_Y     = 8
NUM_CHANNELS= 64
ACTION_SIZE = BOARD_X * BOARD_Y  # 64
DROPOUT     = 0.3
LR          = 0.001

# Generate synthetic Othello board states and policy/value targets
np.random.seed(42)
N_SAMPLES = 5000
X_boards  = np.random.choice([-1, 0, 1], size=(N_SAMPLES, BOARD_X, BOARD_Y)).astype('float32')
# policy: probability distribution over actions
pi_raw    = np.random.dirichlet(np.ones(ACTION_SIZE), size=N_SAMPLES).astype('float32')
# value: scalar in [-1, 1]
v_raw     = np.random.uniform(-1, 1, size=(N_SAMPLES, 1)).astype('float32')

split = int(0.8 * N_SAMPLES)
X_tr, X_te = X_boards[:split], X_boards[split:]
pi_tr, pi_te = pi_raw[:split], pi_raw[split:]
v_tr,  v_te  = v_raw[:split],  v_raw[split:]
print(f'Synthetic Othello data: train={X_tr.shape} test={X_te.shape}')

In [ ]:
def build_smelly_othello_net():
    """yudasong OthelloNNet.py — SMELL: Flatten() then Dense(1024)"""
    inp     = Input(shape=(BOARD_X, BOARD_Y))
    x       = tf.keras.layers.Reshape((BOARD_X, BOARD_Y, 1))(inp)
    x = Activation('relu')(BatchNormalization(axis=3)(Conv2D(NUM_CHANNELS, 3, padding='same')(x)))
    x = Activation('relu')(BatchNormalization(axis=3)(Conv2D(NUM_CHANNELS, 3, padding='same')(x)))
    x = Activation('relu')(BatchNormalization(axis=3)(Conv2D(NUM_CHANNELS, 3, padding='valid')(x)))
    x = Activation('relu')(BatchNormalization(axis=3)(Conv2D(NUM_CHANNELS, 3, padding='valid')(x)))
    # ❌ SMELL (line 25): Flatten then Dense — large weight matrix
    x_flat  = Flatten()(x)
    s_fc1   = Dropout(DROPOUT)(Activation('relu')(BatchNormalization(axis=1)(Dense(1024)(x_flat))))
    s_fc2   = Dropout(DROPOUT)(Activation('relu')(BatchNormalization(axis=1)(Dense(512)(s_fc1))))
    pi_out  = Dense(ACTION_SIZE, activation='softmax', name='pi')(s_fc2)
    v_out   = Dense(1,           activation='tanh',    name='v')(s_fc2)
    model = Model(inputs=inp, outputs=[pi_out, v_out])
    model.compile(loss=['categorical_crossentropy', 'mean_squared_error'],
                  optimizer=Adam(LR))
    return model

def build_clean_othello_net():
    """Fixed: GlobalAveragePooling2D replaces Flatten()"""
    inp = Input(shape=(BOARD_X, BOARD_Y))
    x   = tf.keras.layers.Reshape((BOARD_X, BOARD_Y, 1))(inp)
    x = Activation('relu')(BatchNormalization(axis=3)(Conv2D(NUM_CHANNELS, 3, padding='same')(x)))
    x = Activation('relu')(BatchNormalization(axis=3)(Conv2D(NUM_CHANNELS, 3, padding='same')(x)))
    x = Activation('relu')(BatchNormalization(axis=3)(Conv2D(NUM_CHANNELS, 3, padding='valid')(x)))
    x = Activation('relu')(BatchNormalization(axis=3)(Conv2D(NUM_CHANNELS, 3, padding='valid')(x)))
    # ✅ FIX: GlobalAveragePooling2D — compact spatial summary
    x_gap   = GlobalAveragePooling2D()(x)
    s_fc1   = Dropout(DROPOUT)(Activation('relu')(Dense(256)(x_gap)))
    s_fc2   = Dropout(DROPOUT)(Activation('relu')(Dense(128)(s_fc1)))
    pi_out  = Dense(ACTION_SIZE, activation='softmax', name='pi')(s_fc2)
    v_out   = Dense(1,           activation='tanh',    name='v')(s_fc2)
    model = Model(inputs=inp, outputs=[pi_out, v_out])
    model.compile(loss=['categorical_crossentropy', 'mean_squared_error'],
                  optimizer=Adam(LR))
    return model

print('Model builders ready')

## BEFORE — Smell Active (Flatten → Dense(1024))

In [ ]:
results_before = []
es = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

for run in range(1, N_RUNS + 1):
    print(f'\n[BEFORE] Run {run}/{N_RUNS}')
    K.clear_session(); gc.collect()
    tracker = EmissionsTracker(
        project_name=f'OthelloNNet_before_run{run}',
        save_to_file=False, log_level='error'
    )
    tracker.start()
    model = build_smelly_othello_net()
    history = model.fit(X_tr, [pi_tr, v_tr], epochs=EPOCHS, batch_size=BATCH,
                        validation_data=(X_te, [pi_te, v_te]),
                        callbacks=[es], verbose=1)
    tracker.stop()
    em = tracker.final_emissions_data
    results_before.append({
        'run': run, 'epochs_run': len(history.history['loss']),
        'energy_kWh': round(em.energy_consumed,8),
        'cpu_energy_kWh': round(em.cpu_energy,8),
        'gpu_energy_kWh': round(em.gpu_energy,8),
        'ram_energy_kWh': round(em.ram_energy,8),
        'emissions_kgCO2': round(em.emissions,8),
        'duration_s': round(em.duration,2),
    })
    print(f'  energy={em.energy_consumed:.6f} kWh  CO2={em.emissions:.6f} kg  t={em.duration:.1f}s')
    del model; gc.collect()

df_before = pd.DataFrame(results_before)
df_before.to_csv('results/yudasong_OthelloNNet_before.csv', index=False)
print('\n--- BEFORE means ---')
print(df_before.mean(numeric_only=True))
print('Saved results/yudasong_OthelloNNet_before.csv')

## AFTER — Smell Fixed (GlobalAveragePooling2D → Dense)

In [ ]:
results_after = []

for run in range(1, N_RUNS + 1):
    print(f'\n[AFTER] Run {run}/{N_RUNS}')
    K.clear_session(); gc.collect()
    tracker = EmissionsTracker(
        project_name=f'OthelloNNet_after_run{run}',
        save_to_file=False, log_level='error'
    )
    tracker.start()
    model = build_clean_othello_net()
    history = model.fit(X_tr, [pi_tr, v_tr], epochs=EPOCHS, batch_size=BATCH,
                        validation_data=(X_te, [pi_te, v_te]),
                        callbacks=[es], verbose=1)
    tracker.stop()
    em = tracker.final_emissions_data
    results_after.append({
        'run': run, 'epochs_run': len(history.history['loss']),
        'energy_kWh': round(em.energy_consumed,8),
        'cpu_energy_kWh': round(em.cpu_energy,8),
        'gpu_energy_kWh': round(em.gpu_energy,8),
        'ram_energy_kWh': round(em.ram_energy,8),
        'emissions_kgCO2': round(em.emissions,8),
        'duration_s': round(em.duration,2),
    })
    print(f'  energy={em.energy_consumed:.6f} kWh  CO2={em.emissions:.6f} kg  t={em.duration:.1f}s')
    del model; gc.collect()

df_after = pd.DataFrame(results_after)
df_after.to_csv('results/yudasong_OthelloNNet_after.csv', index=False)
print('\n--- AFTER means ---')
print(df_after.mean(numeric_only=True))
print('Saved results/yudasong_OthelloNNet_after.csv')

In [ ]:
print('\n=== SUMMARY: yudasong/Reinforcement-Learning-Branch-and-Bound ===')
print(f"BEFORE avg energy : {df_before['energy_kWh'].mean():.6f} kWh")
print(f"AFTER  avg energy : {df_after['energy_kWh'].mean():.6f} kWh")
print(f"BEFORE avg CO2    : {df_before['emissions_kgCO2'].mean():.6f} kg")
print(f"AFTER  avg CO2    : {df_after['emissions_kgCO2'].mean():.6f} kg")
print(f"BEFORE avg time   : {df_before['duration_s'].mean():.1f} s")
print(f"AFTER  avg time   : {df_after['duration_s'].mean():.1f} s")